# Fluorescence phase response

This notebook reproduces the source analysis without reinterpreting replicate independence. Cells are pooled by strain, phase and replicate, including reused labels within a phase. Exponential includes 18 h; stationary includes 24 h. The arithmetic-mean ratio and the OLS contrast on log2 replicate means are distinct estimates. Demographs are outside this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "Arial", "text.usetex": False})

MESH_INTENSITY_FEATURE = "C2_cell_mesh_intensity_mean"
book = DATASETS / "Dataset S7.xlsx"
phase_df = pd.read_excel(book, sheet_name="Single_cells")
reference = pd.read_excel(book, sheet_name="Plotted_results")

## Recalculate replicate means, phase ratios, and planned contrasts
The grouping, OLS model, eligibility rules, contrasts and Holm correction are retained from the source notebook.

In [ ]:
def calculate_unpaired_log2_fold_change(df, feature=MESH_INTENSITY_FEATURE, strain_col='strain', phase_col='phase', replicate_col='replicate'):
    """
    Calculate log2 fold change between independently generated
    exponential- and stationary-phase biological replicates.

    Calculation hierarchy:
        cells -> biological replicate mean -> phase mean

    For each strain:

        log2FC = log2(
            mean stationary replicate intensity
            /
            mean exponential replicate intensity
        )

    Also calculates SD and SEM between biological replicates.

    Returns
    -------
    replicate_means : pd.DataFrame
        One mean intensity value per strain, phase and biological replicate.

    summary : pd.DataFrame
        Phase means, SD, SEM, replicate numbers and log2 fold change
        for each strain.
    """
    required = [strain_col, phase_col, replicate_col, feature]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    work = df.copy()
    work[feature] = pd.to_numeric(work[feature], errors='coerce')
    work = work[work[phase_col].isin(['Exponential', 'Stationary']) & work[feature].notna() & np.isfinite(work[feature]) & work[replicate_col].notna()].copy()
    replicate_means = work.groupby([strain_col, phase_col, replicate_col], as_index=False)[feature].mean().rename(columns={feature: 'replicate_mean_intensity'})
    results = []
    for strain in replicate_means[strain_col].unique():
        strain_data = replicate_means[replicate_means[strain_col] == strain]
        exp_values = strain_data.loc[strain_data[phase_col] == 'Exponential', 'replicate_mean_intensity'].dropna().to_numpy(dtype=float)
        stat_values = strain_data.loc[strain_data[phase_col] == 'Stationary', 'replicate_mean_intensity'].dropna().to_numpy(dtype=float)
        if len(exp_values) == 0 or len(stat_values) == 0:
            continue
        exp_mean = np.mean(exp_values)
        stat_mean = np.mean(stat_values)
        if len(exp_values) >= 2:
            exp_sd = np.std(exp_values, ddof=1)
            exp_sem = exp_sd / np.sqrt(len(exp_values))
        else:
            exp_sd = np.nan
            exp_sem = np.nan
        if len(stat_values) >= 2:
            stat_sd = np.std(stat_values, ddof=1)
            stat_sem = stat_sd / np.sqrt(len(stat_values))
        else:
            stat_sd = np.nan
            stat_sem = np.nan
        if exp_mean > 0 and stat_mean > 0:
            log2fc = np.log2(stat_mean / exp_mean)
        else:
            log2fc = np.nan
        if np.isfinite(exp_sem) and np.isfinite(stat_sem) and (exp_mean > 0) and (stat_mean > 0):
            propagated_sem = 1 / np.log(2) * np.sqrt((stat_sem / stat_mean) ** 2 + (exp_sem / exp_mean) ** 2)
        else:
            propagated_sem = np.nan
        results.append({strain_col: strain, 'exponential_mean': exp_mean, 'exponential_sd': exp_sd, 'exponential_sem': exp_sem, 'n_exponential_replicates': len(exp_values), 'stationary_mean': stat_mean, 'stationary_sd': stat_sd, 'stationary_sem': stat_sem, 'n_stationary_replicates': len(stat_values), 'log2_fold_change': log2fc, 'log2fc_sem': propagated_sem})
    summary = pd.DataFrame(results)
    if summary.empty:
        return (replicate_means, summary)
    summary['_control_order'] = summary[strain_col].astype(str).str.casefold().ne('control')
    summary = summary.sort_values(['_control_order', 'log2_fold_change'], ascending=[True, True]).drop(columns='_control_order').reset_index(drop=True)
    return (replicate_means, summary)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path
from patsy import build_design_matrices
from statsmodels.stats.multitest import multipletests
from unittest.mock import patch

def add_phase_response_tests(summary, replicate_means, strain_col='strain', phase_col='phase', replicate_col='replicate', control='Control'):
    """
    Two-factor linear model of log2 biological-replicate mean intensity.

    Planned contrast for each strain:
        (Stationary - Exponential)_strain
        - (Stationary - Exponential)_control

    Existing fold changes and confidence intervals are not modified.
    """
    required = [strain_col, phase_col, replicate_col, 'replicate_mean_intensity']
    missing = [column for column in required if column not in replicate_means]
    if missing:
        raise ValueError(f'Missing columns: {missing}')
    result = summary.copy()
    names = result[strain_col].astype(str)
    if names.duplicated().any():
        raise ValueError('summary must contain one row per strain.')
    controls = names[names.str.casefold().eq(control.casefold())]
    if len(controls) != 1:
        raise ValueError('Exactly one control strain is required.')
    control_name = controls.iloc[0]
    work = replicate_means[required].copy()
    work = work[work[phase_col].isin(['Exponential', 'Stationary'])]
    if work[[strain_col, phase_col, replicate_col]].isna().any().any():
        raise ValueError('Replicate identifiers and group labels cannot be missing.')
    if work.duplicated([strain_col, phase_col, replicate_col]).any():
        raise ValueError('Each strain/phase/biological-replicate combination must have one row.')
    work['_strain'] = work[strain_col].astype(str)
    work['_phase'] = work[phase_col].astype(str)
    work['_intensity'] = pd.to_numeric(work['replicate_mean_intensity'], errors='coerce')
    result['model_log2FC_difference_vs_control'] = np.nan
    result['p_vs_control'] = np.nan
    result['p_holm'] = np.nan
    result['significance'] = 'NT'
    result['test_status'] = 'Not tested'
    eligible = []
    for strain in names:
        sub = work[work['_strain'].eq(strain)]
        counts = sub['_phase'].value_counts()
        valid = np.isfinite(sub['_intensity']) & sub['_intensity'].gt(0)
        if counts.get('Exponential', 0) < 2 or counts.get('Stationary', 0) < 2:
            reason = 'Fewer than two replicates in one or both phases'
        elif not valid.all():
            reason = 'Missing, non-finite, or non-positive replicate intensity'
        else:
            eligible.append(strain)
            reason = 'Eligible'
        result.loc[names.eq(strain), 'test_status'] = reason
    if control_name not in eligible:
        raise ValueError('The control requires at least two positive, finite biological replicate means in each phase.')
    if len(eligible) < 2:
        raise ValueError('No eligible strain is available for comparison.')
    levels = [control_name] + sorted((strain for strain in eligible if strain != control_name))
    model_data = work[work['_strain'].isin(levels)].copy()
    model_data['_strain'] = pd.Categorical(model_data['_strain'], categories=levels)
    model_data['_phase'] = pd.Categorical(model_data['_phase'], categories=['Exponential', 'Stationary'])
    model_data['_log2_intensity'] = np.log2(model_data['_intensity'])
    model = smf.ols('_log2_intensity ~ C(_strain) * C(_phase)', data=model_data).fit()
    if model.df_resid <= 0 or np.std(model.resid) < 1e-12:
        raise ValueError('Insufficient residual variation for statistical testing.')
    for strain in levels[1:]:
        comparison = pd.DataFrame({'_strain': [strain, strain, control_name, control_name], '_phase': ['Stationary', 'Exponential', 'Stationary', 'Exponential']})
        design = np.asarray(build_design_matrices([model.model.data.design_info], comparison)[0])
        contrast = design[0] - design[1] - design[2] + design[3]
        test = model.t_test(contrast)
        mask = names.eq(strain)
        result.loc[mask, 'model_log2FC_difference_vs_control'] = np.asarray(test.effect).item()
        result.loc[mask, 'p_vs_control'] = np.asarray(test.pvalue).item()
        result.loc[mask, 'test_status'] = 'Tested'
    tested = result['p_vs_control'].notna()
    result.loc[tested, 'p_holm'] = multipletests(result.loc[tested, 'p_vs_control'].to_numpy(), method='holm')[1]

    def stars(p):
        if p < 0.0001:
            return '****'
        if p < 0.001:
            return '***'
        if p < 0.01:
            return '**'
        if p < 0.05:
            return '*'
        return 'ns'
    result.loc[tested, 'significance'] = result.loc[tested, 'p_holm'].map(stars)
    result.loc[names.eq(control_name), 'significance'] = ''
    result.loc[names.eq(control_name), 'test_status'] = 'Reference control'
    return (result, model)

In [ ]:
replicate_means, log2fc_summary = calculate_unpaired_log2_fold_change(phase_df, replicate_col="replicate")
log2fc_summary, phase_response_model = add_phase_response_tests(log2fc_summary, replicate_means, replicate_col="replicate")
check = log2fc_summary.set_index("strain").reindex(reference.strain)
np.testing.assert_allclose(check.log2_fold_change, reference.log2_fold_change, rtol=1e-12)
np.testing.assert_allclose(check.p_holm, reference.p_holm, rtol=1e-9, equal_nan=True)

## Bootstrap confidence intervals and figure
Resample the source replicate-group means separately within each phase: 10,000 resamples, seed 42.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def plot_unpaired_log2_fold_change(summary, replicate_means, strain_col='strain', n_boot=10000, random_state=42):
    """Plot log2 fold changes and 95% bootstrap CIs at 70 × 85 mm."""
    plot_df = summary.copy()
    plot_df['_control_order'] = plot_df[strain_col].astype(str).str.casefold() != 'control'
    plot_df = plot_df.sort_values(['_control_order', 'log2_fold_change'], ascending=[True, True]).drop(columns='_control_order').reset_index(drop=True)
    if plot_df.empty:
        raise ValueError('summary contains no strains to plot.')

    def format_label(strain):
        strain = str(strain)
        if strain.casefold() == 'control':
            return 'Control'
        suffix = 'msfTqox2' if strain == 'yggE' else 'msfGFP'
        return f'{strain}-{suffix}'
    labels = [format_label(s) for s in plot_df[strain_col]]
    rng = np.random.default_rng(random_state)
    (ci_lower, ci_upper) = ([], [])
    for strain in plot_df[strain_col]:
        data = replicate_means.loc[replicate_means[strain_col].eq(strain)]
        exp = data.loc[data['phase'].eq('Exponential'), 'replicate_mean_intensity'].to_numpy(dtype=float)
        stat = data.loc[data['phase'].eq('Stationary'), 'replicate_mean_intensity'].to_numpy(dtype=float)
        exp = exp[np.isfinite(exp)]
        stat = stat[np.isfinite(stat)]
        if len(exp) < 2 or len(stat) < 2:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
            continue
        exp_boot = rng.choice(exp, size=(n_boot, len(exp)), replace=True).mean(axis=1)
        stat_boot = rng.choice(stat, size=(n_boot, len(stat)), replace=True).mean(axis=1)
        valid = np.isfinite(exp_boot) & np.isfinite(stat_boot) & (exp_boot > 0) & (stat_boot > 0)
        if valid.sum() < 0.9 * n_boot:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
            continue
        boot_fc = np.log2(stat_boot[valid] / exp_boot[valid])
        (lower, upper) = np.percentile(boot_fc, [2.5, 97.5])
        ci_lower.append(lower)
        ci_upper.append(upper)
    plot_df['ci_lower'] = ci_lower
    plot_df['ci_upper'] = ci_upper
    text_color = '#263C45'
    interval_color = '#365C66'
    point_color = '#237D86'
    control_color = '#788892'
    reference_color = '#A5B0B7'
    with plt.rc_context({'font.family': ['Arial', 'DejaVu Sans'], 'font.size': 5.5, 'axes.linewidth': 0.6}):
        (fig, ax) = plt.subplots(figsize=(70 / 25.4, 85 / 25.4))
        fig.patch.set_facecolor('white')
        ax.set_facecolor('white')
        ax.axvline(0, color=reference_color, linewidth=0.7, linestyle=(0, (2, 2)), zorder=0)
        for (i, row) in plot_df.iterrows():
            lower = row['ci_lower']
            upper = row['ci_upper']
            estimate = row['log2_fold_change']
            if np.isfinite(lower) and np.isfinite(upper):
                ax.hlines(i, lower, upper, color=interval_color, linewidth=0.9, zorder=2)
                ax.vlines([lower, upper], i - 0.08, i + 0.08, color=interval_color, linewidth=0.7, zorder=2)
            if np.isfinite(estimate):
                color = control_color if str(row[strain_col]).casefold() == 'control' else point_color
                ax.scatter(estimate, i, s=18, color=color, edgecolor='white', linewidth=0.5, zorder=3)
        n = len(plot_df)
        ax.set_yticks(np.arange(n), labels=labels)
        for (tick_label, strain) in zip(ax.get_yticklabels(), plot_df[strain_col]):
            if str(strain).casefold() != 'control':
                tick_label.set_fontstyle('italic')
        ax.set_ylim(n - 0.5, -0.5)
        ax.set_xlabel('Log$_2$ fold change\nStationary / Exponential', fontsize=5.5, color=text_color, labelpad=3)
        ax.set_title('Fluorescence change across growth phases', loc='left', fontsize=6.5, fontweight='bold', color=text_color, pad=5)
        ax.spines[['top', 'right', 'left']].set_visible(False)
        ax.spines['bottom'].set_color(reference_color)
        ax.tick_params(axis='y', length=0, pad=3, labelsize=5, labelcolor=text_color)
        ax.tick_params(axis='x', direction='out', length=2, width=0.6, labelsize=5, colors=text_color)
        ax.grid(False)
        fig.text(0.5, 0.025, 'Points: log$_2$ fold change  |  Whiskers: 95% bootstrap CI', ha='center', va='bottom', fontsize=5, color='#73838B')
        fig.subplots_adjust(left=0.4, right=0.98, bottom=0.18, top=0.88)
        plt.show()
    return fig

In [ ]:
from unittest.mock import patch
with patch.object(plt, 'show'):
    fig = plot_unpaired_log2_fold_change(log2fc_summary, replicate_means, n_boot=10000, random_state=42)
ax = fig.axes[0]
ordered = log2fc_summary.assign(_control_order=lambda frame: frame['strain'].astype(str).str.casefold().ne('control')).sort_values(['_control_order', 'log2_fold_change']).reset_index(drop=True)
fig.subplots_adjust(right=0.86, bottom=0.22)
for (i, row) in ordered.iterrows():
    annotation = row['significance']
    if annotation.startswith('*') or annotation == 'NT':
        ax.text(1.03, i, annotation, transform=ax.get_yaxis_transform(), ha='left', va='center', fontsize=5.5, fontweight='bold', clip_on=False)
footer = 'Whiskers: original 95% bootstrap CI\nStars: Holm-adjusted strain × phase contrasts\n* <0.05   ** <0.01   *** <0.001   **** <0.0001'
existing_footer = next((text for text in fig.texts if 'Whiskers' in text.get_text()), None)
if existing_footer is not None:
    existing_footer.set_text(footer)
    existing_footer.set_fontsize(5)
    existing_footer.set_position((0.5, 0.015))
else:
    fig.text(0.5, 0.015, footer, ha='center', va='bottom', fontsize=5)
plt.show()
display(log2fc_summary[['strain', 'log2_fold_change', 'model_log2FC_difference_vs_control', 'p_vs_control', 'p_holm', 'test_status']])